# Autoresearch MLflow Stock Model

- Dataset: `data/runs/phase2-source-20260424`
- Evaluator: `scripts/autoresearch_eval.py`
- MLflow tracking URI: `sqlite:///data/mlflow/mlflow.db`
- Primary metric: `sharpe_diff`
- Secondary metrics: `information_ratio`, `active_return`, `sharpe_diff_ci_low`
- Validation: fixed walk-forward portfolio backtest with SPY date alignment

The generic shuffled 5-fold CV template is intentionally not used for the final comparison here because it would break the portfolio time-ordering and SPY-relative evaluation contract. The repo's walk-forward backtest is the validation authority for this experiment.


In [ ]:
from pathlib import Path

import pandas as pd

RESULTS_TSV = Path("experiments/autoresearch/results.tsv")
results = pd.read_csv(RESULTS_TSV, sep="\t")
results.tail()


In [ ]:
experiment_rows = results.loc[
    results["iteration_id"].astype(str).str.startswith("exp_"),
    [
        "candidate_id",
        "candidate_sharpe",
        "spy_sharpe",
        "sharpe_diff",
        "active_return",
        "information_ratio",
        "sharpe_diff_ci_low",
        "mean_turnover",
        "status",
    ],
].copy()
leaderboard = experiment_rows.sort_values(
    ["sharpe_diff", "information_ratio"], ascending=False
).reset_index(drop=True)
leaderboard.head(10)


## Leaderboard Snapshot

| Candidate | Sharpe | SPY Sharpe | Sharpe Diff | Active Return | IR | CI Low | Turnover | Status |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | --- |
| `e8_weight_ridge_1p2_lgbm_0p8_scale_1p20` | 3.255864 | 1.130685 | 2.125178 | 0.729631 | 2.313480 | -0.335028 | 0.682642 | provisional |
| `e8_weight_ridge_1p2_lgbm_0p8_scale_1p10` | 3.255394 | 1.130685 | 2.124709 | 0.706076 | 2.336423 | -0.351793 | 0.679460 | provisional |
| `e8_weight_ridge_1p2_lgbm_0p8_scale_1p30` | 3.253273 | 1.130685 | 2.122588 | 0.750389 | 2.291243 | -0.342935 | 0.683211 | provisional |
| `e8_weight_ridge_1p2_lgbm_0p8_scale_1p15` | 3.253133 | 1.130685 | 2.122448 | 0.717022 | 2.323494 | -0.343275 | 0.681167 | provisional |
| `e8_weight_ridge_1p2_lgbm_0p8_scale_1p05` | 3.241327 | 1.130685 | 2.110641 | 0.691341 | 2.341578 | -0.366302 | 0.678875 | provisional |


## Winner

Winner model: `e8_weight_ridge_1p2_lgbm_0p8_scale_1p20`

Family: Ridge plus LightGBM E8 weighted blend with score calibration

Primary metric: `sharpe_diff = 2.125178`, up from baseline `1.856511`

Secondary metrics: `information_ratio = 2.313480`, up from baseline `2.212435`; `active_return = 0.729631`; `sharpe_diff_ci_low = -0.335028`, still provisional.

Validation detail: 46 aligned SPY observations from the fixed walk-forward evaluator, top 100 assets, 48 requested rebalances, 5 bps transaction costs, optimizer max weight 0.30.

Why it won: reweighting the E8 internal z-scored model legs toward Ridge (`1.2`) versus LightGBM (`0.8`) and scaling the final score by `1.20` improved the optimizer's risk-adjusted allocation without changing benchmark alignment or evaluator logic.

CatBoost result: CatBoost was added and tested as standalone return regression, rank-label regression, top-tercile classification, Ridge+CatBoost blends, and Ridge+LightGBM+CatBoost ensembles. The best CatBoost candidate that passed the objective gate was `e8_catboost_1p3_0p7_0p2_scale_1p20` with `sharpe_diff = 2.094075` and `information_ratio = 2.276817`, below the current winner.

Next iteration: run broader confirmation coverage for the current winner and a narrower confirmation sweep around `scale=1.10` to `scale=1.20`; keep CatBoost available for feature/label work, but do not replace the Ridge+LightGBM production path from the current evidence.
